---
title: "Hw 4: RNNs"
author: "Jorge Bris Moreno"
format:
  html:
    toc: true
    embed-resources: true
    code-fold: true
---

# Homework-4
Recurrent Neural Networks  

## Problem-3: 

Write code for a character-based RNN in PyTorch. You can use his source code and any code online to assist. If you do, just reference where you sought help from.

- Choose a large text corpus, such as a collection of novels from [Project Gutenburg](https://www.gutenberg.org/). You can use other data, but you should explain where your data comes from.

**Text Data**: I will use the Harry Potter series by J.K. Rowling. We obtained the seven novels during our bootcamp.

INSERT ANSWER HERE

- Perform any necessary preprocessing, explaining what steps you take. In particular, what forms of normalization do you use? Do you define characters with special meaning?

In [5]:
# INSERT CODE

# import libraries
import os
import re
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import silhouette_samples
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm


Here, we first take out chapter titles, pages' number information, skipping empty lines, and we remove double white spaces to make it single ones. Then, we combine all books into a single string and save it.

In [6]:
# function to identify titles
def Chapter_title_identificator(previous_line, current_line, next_line):
    if previous_line.strip() == '' and next_line.strip() == '':
        if current_line.isupper():
            return True
    return False

# load and preprocess data
def data_preprocessor(file_paths):
    all_text = []
    # utf-8
    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()

        # Filter out lines containing 'Page |'
        lines = [line for line in lines if "Page |" not in line]

        # store lines
        processed_lines = []
        num_lines = len(lines)

        # Iterate through lines
        i = 0
        while i < num_lines:
            # avoid boundaries for titles (empty lines before and after title line)
            if i > 0 and i < num_lines - 1:
                if Chapter_title_identificator(lines[i-1], lines[i], lines[i+1]):
                    # Skip only chapter title line
                    i += 1
                    continue
            # Skip lines that start with '/' 
            if not lines[i].startswith('/'):
                # Skip empty lines
                if lines[i].strip(): 
                    processed_lines.append(lines[i])
            i += 1

        # put lines into a single string
        text = "".join(processed_lines)

        # Removing double spaces
        text = ' '.join(text.split())

        all_text.append(text)

    # All text to one string
    full_text = " ".join(all_text)

    return full_text

# Paths
file_paths = [f"./corpus/hp_{i}.txt" for i in range(1, 8)]

# Preprocess novels
processed_text = data_preprocessor(file_paths)
print(processed_text[:0])

# Save file
with open('processed_text.txt', 'w', encoding='utf-8') as file:
    file.write(processed_text)

Now, we are keeping all the words and only selected symbols that can carry meaning: whitespace, dot, coma, interrogation and exclamation marks, colon and semi-colon, and dash. We are also making all lower case for better interpretation. The reason behind all these choices is that we want to mantain as much information as possible while not utilizing unuseful tokens in training. Then we preprocess the text and we turn characters into integers:

In [ ]:
# read preprocessed text
with open('processed_text.txt', 'r', encoding='utf-8') as file:
    text = file.read()

# make it lower-case
text = text.lower()

# remove all not common alphanumeric characters
allowed_chars = string.ascii_letters + string.digits + " .,!?;:-'" 
text = "".join([c if c in allowed_chars else " " for c in text])

# Characters to integers and vice versa
vocab = sorted(set(text))
character_to_index = {c: i for i, c in enumerate(vocab)}
index_to_character = {i: c for i, c in enumerate(vocab)}

# Convert the text to integers
text_as_integer = [character_to_index[c] for c in text]

- Efficiently load and batch the dataset for training using a `DataLoader`. Make sure to reserve some of the data for validation and testing. Describe how you handle batching and sequence lengths.

Here, we are making the sequences of length 500, so they are long enough to carry meaning while not too large to process. We are batching in 64 sequences. We are also splitting the data into training, validation, and test sets.

In [8]:
# INSERT CODE

# lists to store sequences and targets
sequences = []
targets = []

# sequences
sequence_length = 500
samp_per_epoch = len(text) // (sequence_length + 1)

# batch
batch_count = 64

# dataset
character_tensor = torch.tensor(text_as_integer, dtype=torch.long)

# create sequences and targets
for i in range(0, len(character_tensor) - sequence_length, sequence_length):
    sequences.append(character_tensor[i:i+sequence_length])
    targets.append(character_tensor[i+1:i+sequence_length+1])

# create dataset as tensor
dataset = TensorDataset(torch.stack(sequences), torch.stack(targets))

# split into train, val, test
size_train = int(0.8 * len(dataset))
size_val = int(0.2 * len(dataset))
size_test = len(dataset) - size_train - size_val

train_data, val_data, test_data = random_split(dataset, [size_train, size_val, size_test])

# dataloader
train_dataloader = DataLoader(train_data, batch_size=batch_count, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=batch_count)
test_dataloader = DataLoader(test_data, batch_size=batch_count)


- Define your RNN model. Discuss the number of layers, hidden units, and the type of RNN cells you use. What is the total number of parameters in your model? Explain the rationale behind your architectural choices.

The architecture is printed below. The reason to use LSTM cell is that they are better to prevent the vanishing gradient problem. We are also using 2 layers as more will take very long to train in my computer and the size of our corpus is not that big (so we do not want to overfit). I have also chosen 256 hidden units as it seems a good number based on the previous size of layers chosen. I have also used 128 embedding dimension as when trying to increase it my kernel was dying (so it was the maximum I could use).

In [9]:
# RNN 
class RNN(nn.Module):
    # Initialize model
    def __init__(self, size_vocabulary, embedding, hidden_unit_count, layer_count):
        # Call parent function
        super(RNN, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(size_vocabulary, embedding)
        
        # RNN layer
        self.rnn = nn.LSTM(embedding, hidden_unit_count, layer_count, batch_first=True)
        
        # output
        self.fc = nn.Linear(hidden_unit_count, size_vocabulary)

    # Forward pass
    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.rnn(x)
        x = self.fc(x)
        # return output
        return x

In [10]:
# hyperparameters
size_vocabulary = len(vocab)
num_layers = 2
hidden_dimension = 256
embedding_dimension = 128

# model
model = RNN(size_vocabulary, embedding_dimension, hidden_dimension, num_layers)

# parameter count
def parameter_counter(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# count parameters
param_count = parameter_counter(model)

# print model architecture
display(model)

# print number of parameters
print(f'Total number of parameters: {param_count}')

RNN(
  (embedding): Embedding(45, 128)
  (rnn): LSTM(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=45, bias=True)
)

Total number of parameters: 938925


- Write the training loop.

In [12]:
# Parameters
total_epochs = 10
lr = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

# lists to store losses
train_loss = []
val_loss = []

# Training loop
for epoch in range(total_epochs):
    
    # Training
    model.train()

    # initialize total training loss
    total_train_loss = 0.0
    
    # Training loop
    train_loop = tqdm(enumerate(train_dataloader), total=len(train_dataloader), leave=False)

    for batch_index, (data, target) in train_loop:
        optimizer.zero_grad()
        output = model(data)
        output = output.permute(0, 2, 1)
        
        # calculate loss
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        # add loss to total training loss
        total_train_loss += loss.item()
        # update progress bar
        train_loop.set_description(f"Epoch [{epoch+1}/{total_epochs}]")
        train_loop.set_postfix(loss=loss.item())
    
    # avg training loss and perplexity per epoch
    avg_train_loss = total_train_loss / len(train_dataloader)
    train_loss.append(avg_train_loss)
    train_perplexity = torch.exp(torch.tensor(avg_train_loss)).item()
    
    # Validation initialization
    model.eval()
    total_val_loss = 0.0
    val_loop = tqdm(enumerate(val_dataloader), total=len(val_dataloader), leave=False)
    
    # Validation
    with torch.no_grad():
        for batch_index, (data, target) in val_loop:
            output = model(data)
            output = output.permute(0, 2, 1)
            # Calculate loss
            loss = criterion(output, target)
            total_val_loss += loss.item()
            # Update progress bar
            val_loop.set_description(f"Validation Epoch [{epoch+1}/{total_epochs}]")
            val_loop.set_postfix(val_loss=loss.item())
    
    # Avg validation loss and perplexity per epoch
    avg_val_loss = total_val_loss / len(val_dataloader)
    val_loss.append(avg_val_loss)
    val_perplexity = torch.exp(torch.tensor(avg_val_loss)).item()
    
    # Epoch end summary
    print(f"Epoch {epoch+1}/{total_epochs}, Train Loss: {avg_train_loss:.4f}, Train Perplexity: {train_perplexity:.4f}, Val Loss: {avg_val_loss:.4f}, Val Perplexity: {val_perplexity:.4f}")


Epoch 1/10, Train Loss: 1.7381, Train Perplexity: 5.6867, Val Loss: 1.4618, Val Perplexity: 4.3136


Epoch 2/10, Train Loss: 1.3720, Train Perplexity: 3.9432, Val Loss: 1.3172, Val Perplexity: 3.7328


Epoch 3/10, Train Loss: 1.2841, Train Perplexity: 3.6114, Val Loss: 1.2641, Val Perplexity: 3.5400


Epoch 4/10, Train Loss: 1.2399, Train Perplexity: 3.4552, Val Loss: 1.2320, Val Perplexity: 3.4281


Epoch 5/10, Train Loss: 1.2116, Train Perplexity: 3.3588, Val Loss: 1.2111, Val Perplexity: 3.3573


Epoch 6/10, Train Loss: 1.1901, Train Perplexity: 3.2873, Val Loss: 1.1940, Val Perplexity: 3.3001


Epoch 7/10, Train Loss: 1.1743, Train Perplexity: 3.2357, Val Loss: 1.1853, Val Perplexity: 3.2715


Epoch 8/10, Train Loss: 1.1625, Train Perplexity: 3.1979, Val Loss: 1.1719, Val Perplexity: 3.2282


Epoch 9/10, Train Loss: 1.1503, Train Perplexity: 3.1591, Val Loss: 1.1622, Val Perplexity: 3.1969


Epoch 10/10, Train Loss: 1.1418, Train Perplexity: 3.1323, Val Loss: 1.1567, Val Perplexity: 3.1794



- Monitor and report on the training progress by tracking the loss. After training, evaluate the model's performance using a suitable evaluation metric (e.g., perplexity) on a validation dataset or a held-out portion of the training data. Discuss the results.

This was done above (as I run the training-loop right after defining it). Throughout the ten epochs trained, the loss has gotten better for both, validation and training, and even more notizable for the perplexity. However, it seems that for the last three epochs the improvement hasn't been too large, it seems that it was settling down. The training loss seems decently low for both sets, and similar values which indicates that there shouldn't be much overfitting. Moreover, the perplexity 


- Specify the hyper-parameters (for example, model hyper-parameters, as well as sampling size and beam width from below) used in your model. Find suitable settings using a validation split.

I have used the following hyperparameters:

- Number of epochs: 10 (more epochs would take too long to train and it seems that the model is already settling down)
- Learning rate: 0.01 (it seems to be a good value for the model, and smaller will take too long for my laptop to train)
- Batch size: 64 (it seems to be a good value for the model)

Notes from before:

- Sequence length: 500 
- Number of hidden units: 256
- Number of layers: 2
- Embedding dimension: 128


- Implement a text generation function using the trained RNN model. Provide a prompt or seed text, and use the RNN to generate a sequence of characters. Experiment with different prompt texts and observe how the generated text changes. Discuss any interesting patterns or observations you make during the text generation process.

Here we created the function and ran it for some random prompts I came up with (some Harry Potter related to test if it would work better for these).

In [18]:
# INSERT CODE

# function to generate text
def text_generator(model, start_sentence, length, temperature=1.0):

    # evaluation mode
    model.eval()

    # convert start sentence to tensor
    input_eval = [character_to_index[s] for s in start_sentence]
    input_eval = torch.tensor(input_eval, dtype=torch.long).unsqueeze(0)

    # generated text list
    generated_text = []

    with torch.no_grad():
        
        # loop through the length specified
        for i in range(length):

            # get the output
            output = model(input_eval)
            output = output[:, -1, :] / temperature  
            probabilities = nn.functional.softmax(output, dim=-1)
            
            # sample the next character
            predicted_id = torch.multinomial(probabilities, num_samples=1)
            
            # add the predicted character to the generated text
            input_eval = torch.cat([input_eval, predicted_id], dim=1)

            # convert the predicted character to a string and append to the generated text
            generated_text.append(index_to_character[predicted_id.item()])

    # return text
    return start_sentence + ''.join(generated_text)

# Use the funtion to generate text
# My Prompts
prompts = ["is it", "harry potter and", "yesterday i went to", "harry, listen to me", "once upon a time", "the end of the world is near", "harry, i want", "ron and i", "the dark lord", "i am a wizard", "my dear harry", "the forbidden forest", "the master of", "only if you", "the ball was", "make me some"]

# generate text for each prompt
for prompt in prompts:
    print(f"prompt: {prompt}")
    print(text_generator(model, prompt, 100))
    print('-' * 100)


prompt: is it
is it. he could, i used sirius was not without spotty grafficance much fifty- water if harry and harry wa
----------------------------------------------------------------------------------------------------
prompt: harry potter and
harry potter and the walls, portrait metsier across the unucularly curtains?   good loudly,  said sirius and who did
----------------------------------------------------------------------------------------------------
prompt: yesterday i went to
yesterday i went to get thousand over the magoss us anymore, but your mother much, ever insisting red leg in the worlin
----------------------------------------------------------------------------------------------------
prompt: harry, listen to me
harry, listen to me!  said ron s face cleaning explosing and pretending from a moment where all heavencing when she cro
----------------------------------------------------------------------------------------------------
prompt: once upon a time
once upon a t

We see that while some words make sense, the longer you go in the sentence, the less sense it makes. This is because the model is not able to understand the context of the words, and it is just predicting the next character based on the previous ones. This is a common problem with character-based RNNs. However, it is interesting to see the model generating some Harry Potter related content in some scenarios. Some sentences (the begginings since the endings stop makling sense) seem to be taken straight from the books, while others are just random words. An example of this is when I prompted "Harry, listen to me" and the model mentioned Ron, or "make me some" and it mentions ron again. I just found it interesting how much the model is affected by the training data.